<!-- beginner-banner-v2 -->

> 🧭 <strong>비개발자 수강생 안내</strong> — 이 노트북에서 새로 배우는 것: OpenAI 비용 부담을 줄이는 <strong>무료 LLM 대체 경로</strong> — Ollama(Qwen3) / Groq.
>
> - 📖 강의 페이지: <a href="https://siapapa.github.io/appendix/free-llm-ollama/" target="_blank" rel="noopener noreferrer">appendix/free-llm-ollama</a>
> - 🆕 처음이라면 → <a href="https://siapapa.github.io/beginners-guide/" target="_blank" rel="noopener noreferrer">비개발자 학습 가이드</a>
> - 🔤 모르는 단어 → <a href="https://siapapa.github.io/appendix/glossary/" target="_blank" rel="noopener noreferrer">용어 사전</a>
> - 🛠️ 환경/접속 막힘 → <a href="https://siapapa.github.io/setup/" target="_blank" rel="noopener noreferrer">사전 준비</a> · <a href="https://siapapa.github.io/appendix/troubleshooting/" target="_blank" rel="noopener noreferrer">트러블슈팅</a>
>
> 외부 링크는 새 탭으로 열리도록 설정돼 있어 Colab 의 리디렉션 경고 페이지를 거치지 않습니다.<br/>
> <strong>셀은 위에서 아래로 차례대로 실행</strong>하세요. 시연용 코드(<code>구경만 하세요</code> 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 99. 무료 LLM · Ollama (Qwen3) 실습
> 부록 · 소요 약 60분

## 학습 목표

- Colab T4 GPU 위에서 **Ollama** 를 띄우고 **Qwen3** 모델을 받아 OpenAI 호환 API로 호출한다.
- LlamaIndex / LangChain / Vanna 의 LLM 초기화 한두 줄만 바꿔 강의 노트북을 **무료**로 그대로 돌릴 수 있게 한다.
- 비교: **Groq 무료 Tier**(OpenAI 호환·CPU OK)로 전환하는 패턴을 익힌다.
- 한계 체감: 응답 속도·SQL 정확도가 OpenAI 대비 어떻게 다른지 정량 비교한다.

> 본 노트북은 강의 노트북 `00`~`19`의 **대체 LLM 경로**를 확인하기 위한 부록입니다. 병원 DB(`01_postgres_basics` 적재) 를 사용합니다.

## 사전 체크 — 런타임 GPU 확인

Colab 상단 메뉴: **런타임 > 런타임 유형 변경 > 하드웨어 가속기 → T4 GPU**.

GPU가 없으면 **경로 B (Groq)** 부분만 실행해도 본 노트북의 후반부 비교 실습이 진행됩니다.

In [ ]:
# GPU 사용 가능 여부 확인 (없어도 Groq 경로로 진행 가능)
import subprocess
gpu_ok = False
try:
    out = subprocess.check_output(["nvidia-smi", "-L"], text=True, stderr=subprocess.STDOUT)
    print(out.strip())
    gpu_ok = "GPU" in out
except Exception as e:
    print("GPU 미감지:", e)
print("\nGPU available:", gpu_ok)

## Step 0 — 환경 변수 로드

OpenAI 비교 베이스라인이나 Groq 경로 실습을 위해 키를 (있는 경우) 읽어둡니다. 없어도 Ollama 부분만으로도 노트북은 끝까지 진행됩니다.

In [ ]:
import os

def _load_secret(key: str) -> str | None:
    if os.environ.get(key):
        return os.environ[key]
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(key)
        if v:
            os.environ[key] = v
            return v
    except Exception:
        pass
    return None

OPENAI_KEY = _load_secret("OPENAI_API_KEY")    # 비교용 (선택)
GROQ_KEY   = _load_secret("GROQ_API_KEY")      # 경로 B (선택)
NEON_DSN   = _load_secret("NEON_DSN")          # SQL 실습용 (선택)

print("OPENAI_API_KEY:", "set" if OPENAI_KEY else "missing (비교 셀은 자동 스킵)")
print("GROQ_API_KEY  :", "set" if GROQ_KEY   else "missing (경로 B 실습 자동 스킵)")
print("NEON_DSN      :", "set" if NEON_DSN   else "missing (SQL 비교 자동 스킵)")

---

## 경로 A · Step 1 — Ollama 설치 + 백그라운드 서비스

`ollama serve` 는 종료되면 모든 모델 호출이 죽습니다. `nohup` + `subprocess.Popen` 으로 셀이 끝나도 살아 있게 띄우고, `/api/tags` 가 200 OK 를 반환할 때까지 폴링합니다.

In [ ]:
# Ollama 설치 (Colab 한 번)
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ollama serve 를 백그라운드로 — 셀이 끝나도 죽지 않도록 Popen 으로 분리
import os, time, subprocess, requests

LOG_PATH = "/content/ollama.log"

# 이미 실행 중이면 그대로 사용
def _serve_alive() -> bool:
    try:
        requests.get("http://localhost:11434/api/tags", timeout=1)
        return True
    except Exception:
        return False

if not _serve_alive():
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open(LOG_PATH, "w"),
        stderr=subprocess.STDOUT,
        env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    )
    for _ in range(40):
        if _serve_alive():
            break
        time.sleep(1)

assert _serve_alive(), f"ollama serve 가 시작되지 않았습니다. {LOG_PATH} 를 확인하세요."
print("ollama serve ready (http://localhost:11434)")

## 경로 A · Step 2 — Qwen3 모델 + 임베딩 모델 pull

- `qwen3:4b` : T4 에서 빠른 응답, 강의 Easy~Medium 질문에 충분
- `qwen3:8b` : 정확도 한 단계 위, Medium~Hard JOIN/CTE 권장
- `bge-m3`   : 다국어 임베딩 (한국어 OK)

처음 한 번은 모델 다운로드로 3~6분이 걸립니다.

In [ ]:
# 모델 받기 — 메모리 여유에 따라 8b 도 가능
!ollama pull qwen3:4b
!ollama pull bge-m3
!ollama list

## 경로 A · Step 3 — OpenAI SDK 그대로 호출

Ollama 는 `http://localhost:11434/v1/chat/completions` 에서 **OpenAI 호환 엔드포인트**를 제공합니다. 그래서 `openai` 패키지를 그대로 쓰고 `base_url`만 바꾸면 됩니다.

In [ ]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",   # Ollama 는 키를 검증하지 않음 — 아무 문자열이나 OK
)

resp = ollama_client.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": "당신은 PostgreSQL SQL 전문가입니다. 응답은 SQL 한 문장만 출력하세요."},
        {"role": "user",   "content": "patients 테이블에서 남성 환자 수를 세는 SQL?"},
    ],
    temperature=0,
)
print(resp.choices[0].message.content)

!!! note "Qwen3의 `<think>` 태그"
    Qwen3 는 기본적으로 추론 과정을 `<think>...</think>` 로 노출합니다. SQL 만 깔끔히 받고 싶다면 시스템 프롬프트에 "Respond with the final answer only" 를 명시하거나, 아래 LangChain `ChatOllama(reasoning=False)` 옵션을 쓰세요.

---

## 경로 A · Step 4 — LlamaIndex 와 함께 (Text-to-SQL)

`Settings.llm` 만 `OpenAILike` 로 바꾸면 `04`~`08` 노트북의 LlamaIndex 코드가 그대로 굴러갑니다.

In [ ]:
%pip install -q llama-index llama-index-llms-openai-like llama-index-embeddings-ollama \
    sqlalchemy psycopg2-binary pandas

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = OpenAILike(
    model="qwen3:4b",
    api_base="http://localhost:11434/v1",
    api_key="ollama",
    is_chat_model=True,
    is_function_calling_model=False,   # 로컬 양자화 모델은 도구호출 불안정
    temperature=0,
    request_timeout=180.0,
)
Settings.embed_model = OllamaEmbedding(
    model_name="bge-m3",
    base_url="http://localhost:11434",
)
print("LlamaIndex Settings → Ollama (qwen3:4b + bge-m3)")

In [ ]:
# Neon 병원 DB 가 적재되어 있다면 Text-to-SQL 시연 — 없으면 자동 스킵
if NEON_DSN:
    from sqlalchemy import create_engine
    from llama_index.core import SQLDatabase
    from llama_index.core.query_engine import NLSQLTableQueryEngine

    engine = create_engine(NEON_DSN)
    sql_db = SQLDatabase(
        engine,
        include_tables=["patients", "doctors", "visits", "diagnoses", "departments"],
    )
    nlq = NLSQLTableQueryEngine(
        sql_database=sql_db,
        tables=["patients", "doctors", "visits", "diagnoses", "departments"],
    )
    r = nlq.query("진료과별 의사 수를 알려주세요.")
    print("A:  ", r.response)
    print("SQL:", r.metadata.get("sql_query", ""))
else:
    print("NEON_DSN 미설정 — SQL 실습 셀은 스킵합니다. 위 chat 호출만으로도 Ollama 동작은 확인했습니다.")

---

## 경로 A · Step 5 — LangChain `ChatOllama` 로 같은 질문 해보기

`12`~`17` 노트북에서 쓰는 LangChain 경로입니다. `reasoning=False` 로 Qwen3 의 thinking 출력을 끕니다.

In [ ]:
%pip install -q langchain langchain-ollama

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434",
    temperature=0,
    reasoning=False,   # <think> 블록 비활성화
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 PostgreSQL SQL 전문가입니다. SQL만 응답합니다."),
    ("user",   "{question}"),
])
chain = prompt | llm
out = chain.invoke({"question": "visits 테이블에서 status='completed' 인 건수만 세는 SQL?"})
print(out.content)

---

## 경로 A · Step 6 — Vanna.ai 를 Ollama 로

Vanna 는 LLM 어댑터를 다중상속으로 합치는 패턴입니다. `vanna.ollama.Ollama` 어댑터가 내장돼 있어 OpenAI 자리에 그대로 끼웁니다.

In [ ]:
%pip install -q "vanna[chromadb,ollama]" 

In [ ]:
from vanna.ollama import Ollama
from vanna.chromadb import ChromaDB_VectorStore

class MyVanna(ChromaDB_VectorStore, Ollama):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        Ollama.__init__(self, config=config)

vn = MyVanna(config={
    "model": "qwen3:4b",
    "ollama_host": "http://localhost:11434",
})

# 최소 학습: DDL 1개 주입 → 같은 질문을 던져 본다
vn.train(ddl="""
CREATE TABLE patients (
    patient_id  SERIAL PRIMARY KEY,
    name        TEXT,
    gender      TEXT,           -- 'M' / 'F'
    birth_date  DATE
);
""")
sql = vn.generate_sql("남성 환자 수는?")
print("Vanna→SQL:", sql)

---

## 경로 B · Step 1 — Groq 무료 Tier (CPU 런타임에서도 동작)

GPU가 없거나 더 큰 모델이 필요할 때 쓰는 백업 경로. **OpenAI 호환 API**라 코드 변경은 사실상 두 줄(`base_url`, `api_key`).

In [ ]:
if not GROQ_KEY:
    print("GROQ_API_KEY 가 없어 경로 B 셀은 스킵합니다. console.groq.com 에서 발급 후 Colab Secrets 에 등록하세요.")
else:
    from openai import OpenAI as _OpenAI
    groq = _OpenAI(api_key=GROQ_KEY, base_url="https://api.groq.com/openai/v1")

    r = groq.chat.completions.create(
        model="qwen-qwq-32b",  # 또는 "llama-3.3-70b-versatile"
        messages=[
            {"role": "system", "content": "PostgreSQL SQL 전문가. SQL만 응답."},
            {"role": "user",   "content": "최근 30일 동안 visits.status='completed' 인 건수를 진료과별로 보여주는 SQL"},
        ],
        temperature=0,
    )
    print(r.choices[0].message.content)

---

## 비교 실습 — 같은 질문, 세 가지 백엔드

세 경로(OpenAI / Ollama / Groq) 가 **같은 질문**에 대해 어떻게 다른 SQL 을 만들고 얼마나 빠른지 측정합니다. 키가 없는 백엔드는 자동으로 스킵됩니다.

In [ ]:
import time

QUESTIONS = [
    "남성 환자는 몇 명인가요?",
    "진료과별 의사 수를 알려주세요.",
    "각 진료과별로 가장 최근에 진료한 의사의 이름은?",  # Hard
]

def _ask(client, model: str, question: str) -> tuple[str, float]:
    t0 = time.time()
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "당신은 PostgreSQL SQL 전문가입니다. SQL 한 문장만 출력하세요. ```sql 펜스 금지."},
            {"role": "user",   "content": question},
        ],
        temperature=0,
    )
    return r.choices[0].message.content.strip(), time.time() - t0

backends = []
backends.append(("ollama:qwen3:4b", ollama_client, "qwen3:4b"))

if OPENAI_KEY:
    from openai import OpenAI as _O1
    backends.append(("openai:gpt-4o-mini", _O1(api_key=OPENAI_KEY), "gpt-4o-mini"))

if GROQ_KEY:
    from openai import OpenAI as _O2
    backends.append(("groq:qwen-qwq-32b", _O2(api_key=GROQ_KEY, base_url="https://api.groq.com/openai/v1"), "qwen-qwq-32b"))

rows = []
for q in QUESTIONS:
    print("\n=== Q:", q, "===")
    for label, cli, model in backends:
        try:
            ans, dt = _ask(cli, model, q)
            ans_one = " ".join(ans.split())[:140]
            print(f"  [{label:24s}] {dt:5.2f}s  {ans_one}")
            rows.append({"backend": label, "question": q, "seconds": round(dt, 2), "sql": ans_one})
        except Exception as e:
            print(f"  [{label:24s}] ERROR  {type(e).__name__}: {e}")
            rows.append({"backend": label, "question": q, "seconds": None, "sql": f"ERROR: {e}"})

In [ ]:
# 비교표로 정리
import pandas as pd
df = pd.DataFrame(rows)
if not df.empty:
    pivot = df.pivot_table(index="question", columns="backend", values="seconds", aggfunc="first")
    print("응답 시간(초):")
    print(pivot.to_string())

## 비교 결과 해석

- **응답 시간**: OpenAI ≈ Groq < Ollama(GPU 추론). Ollama 는 첫 호출에 모델 로딩 시간 포함.
- **SQL 정확도**: Easy/Medium 까지는 모든 백엔드가 통과. Hard(다중 JOIN + 윈도우 함수) 는 `qwen3:4b` 가 부분 실패하는 경우가 있음 → `qwen3:8b` 또는 Groq 로 승격.
- **비용**: Ollama = 0원, Groq = 무료(분당 RPM 한도), OpenAI = 토큰당 과금.

!!! tip "강의 운영 팁"
    - Day 1·2 (Easy~Medium) → `qwen3:4b` 로 충분.
    - Day 3 Vanna 자가학습 / Day 4 Ragas 채점 → Groq `qwen-qwq-32b` 권장 (속도+정확도).

---

## 실습 과제

1. **본인 질문 5개**를 위 `QUESTIONS` 리스트에 추가해 세 백엔드를 비교하세요.
2. 응답 시간·정확도 표를 정리해 **본인 프로젝트에서 어떤 백엔드를 쓸지** 결론을 내려 보세요.
3. `qwen3:4b` 와 `qwen3:8b` 를 모두 `ollama pull` 한 뒤, 동일 질문에 대해 4b vs 8b 의 정확도 차이를 직접 측정해 보세요. (`!ollama rm qwen3:Xb` 로 디스크 정리 가능)
4. 본인이 가장 즐겨 쓰는 강의 노트북 하나(예: `06_text_to_sql`)에서 LLM 초기화 셀만 위 패턴으로 교체하고 끝까지 동작하는지 확인하세요.

!!! note "핵심 정리"
    - **OpenAI 호환 API** 가 사실상 표준이라, `base_url` + `api_key` 두 줄만 바꾸면 어디든 갈 수 있다.
    - **Ollama**: 완전 무료, GPU 권장, 모델은 `pull` 후 `localhost:11434` 호출.
    - **Groq**: CPU OK, 속도 빠름, 한도 내 무료. 임베딩은 별도(`OllamaEmbeddings` 또는 `HuggingFaceEmbeddings`).
    - **수정 범위**: 노트북당 LLM 초기화 1~2줄. 나머지 강의 코드는 모두 그대로.